# 02_baseline: 新規モデル（TabNet, HistGB, Logistic, NN）のテスト

## 共通設定

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# プロジェクトルートの設定
PROJECT_ROOT = Path(
    "/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026"
)
sys.path.append(str(PROJECT_ROOT))

# 新規モデルのインポート
from common.tabnet.tabnet_model import run_tabnet
from common.histgb.histgb_model import run_histgb
from common.logistic.logistic_model import run_logistic
from common.nn.nn_model import run_nn
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

# 1. 乱数シードの固定
SEED = 42
seed_everything(seed=SEED)

# 2. ターゲット列とID列の設定
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

# 3. パスの設定
DATA_DIR = PROJECT_ROOT / "data" / "input"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / datetime.datetime.now().strftime("%Y%m%d")
SAVED_MODELS_DIR = OUTPUT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 4. ロガーの設定
logger = get_logger(script_name="02_baseline", log_dir=str(OUTPUT_DIR))

# 5. データ読み込み
persona_train = pd.read_csv(DATA_DIR / "employee_persona_train.csv")
persona_test = pd.read_csv(DATA_DIR / "employee_persona_test.csv")
monthly_train = pd.read_csv(DATA_DIR / "employee_monthly_train.csv")
monthly_test = pd.read_csv(DATA_DIR / "employee_monthly_test.csv")

# 6. 簡易的な特徴量集約（月次データ）
def aggregate_monthly(df):
    """monthlyデータを社員IDごとに集約"""
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 集約対象外のカラムを除外
    exclude_cols = [ID_COL, "経過月数"]
    agg_cols = [col for col in num_cols if col not in exclude_cols]
    
    # mean, max, minを計算
    agg_df = df.groupby(ID_COL)[agg_cols].agg(['mean', 'max', 'min']).reset_index()
    agg_df.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in agg_df.columns]
    
    return agg_df

monthly_train_agg = aggregate_monthly(monthly_train)
monthly_test_agg = aggregate_monthly(monthly_test)

# 7. personaとmonthlyを結合
train = persona_train.merge(monthly_train_agg, on=ID_COL, how="left")
test = persona_test.merge(monthly_test_agg, on=ID_COL, how="left")

logger.info("=== [02_baseline] 実験開始 ===")
logger.info(f"結合後 Train Shape: {train.shape}, Test Shape: {test.shape}")

[2026-08-05 08:13:34] [INFO] === [02_baseline] 実験開始 ===
[2026-08-05 08:13:34] [INFO] 結合後 Train Shape: (2761, 74), Test Shape: (2502, 73)


## 基本パラメータ設定

In [2]:
# 全モデル共通のベースパラメータ
base_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
}

## 特徴量エンジニアリング

In [3]:
def build_features(train: pd.DataFrame, test: pd.DataFrame, target_col: str, id_col: str):
    """特徴量エンジニアリングを行う関数"""
    train_proc = train.copy()
    test_proc = test.copy()

    # 非数値列（文字列・日付列）を一時除外
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    
    # 社員ID は後で使うため除外対象から外す
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
        
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")

    # ターゲットとIDを分離
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")
    test_ids = test_proc[id_col]
    
    # NaN値を0で埋める
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    return X_train, y_train, X_test, test_ids

# 特徴量作成
X_train, y_train, X_test, test_ids = build_features(train, test, TARGET_COL, ID_COL)

input_data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

logger.info(f"X_train Shape: {X_train.shape}, X_test Shape: {X_test.shape}")

[2026-08-05 08:13:34] [INFO] X_train Shape: (2761, 57), X_test Shape: (2502, 57)


## TabNet テスト

In [4]:
logger.info("--- TabNet 学習開始 ---")
tabnet_params = base_params.copy()
tabnet_params.update({
    "n_d": 64,
    "n_a": 64,
    "n_steps": 3,
    "gamma": 1.3,
    "lambda_sparse": 1e-3,
    "max_epochs": 50,
    "patience": 10,
    "batch_size": 256,
    "verbose": 0,
})

tabnet_res, _ = run_tabnet(data=input_data, params=tabnet_params)
tabnet_cv = calculate_logloss(input_data["y_train"], tabnet_res["oof_preds"])
logger.info(f"TabNet CV Score: {tabnet_cv:.4f}")

[2026-08-05 08:13:34] [INFO] --- TabNet 学習開始 ---

Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_0_logloss = 0.64868


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_0_logloss = 0.73121


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 32 with best_epoch = 22 and best_val_0_logloss = 0.65353


...


Early stopping occurred at epoch 28 with best_epoch = 18 and best_val_0_logloss = 0.63338


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 34 with best_epoch = 24 and best_val_0_logloss = 0.66297
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models/tabnet_model_fold0.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models/tabnet_model_fold1.zip


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models/tabnet_model_fold2.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models/tabnet_model_fold3.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models/tabnet_model_fold4.zip
[2026-08-05 08:13:52] [INFO] TabNet CV Score: 0.6659


## HistGradientBoosting テスト

In [5]:
logger.info("--- HistGradientBoosting 学習開始 ---")
histgb_params = base_params.copy()
histgb_params.update({
    "learning_rate": 0.05,
    "max_iter": 200,
    "max_depth": 10,
    "min_samples_leaf": 20,
    "l2_regularization": 0.1,
    "max_bins": 255,
})

histgb_res, _ = run_histgb(data=input_data, params=histgb_params)
histgb_cv = calculate_logloss(input_data["y_train"], histgb_res["oof_preds"])
logger.info(f"HistGradientBoosting CV Score: {histgb_cv:.4f}")

[2026-08-05 08:13:52] [INFO] --- HistGradientBoosting 学習開始 ---
[2026-08-05 08:14:01] [INFO] HistGradientBoosting CV Score: 0.6320


## Logistic Regression テスト

In [6]:
logger.info("--- Logistic Regression 学習開始 ---")
logistic_params = base_params.copy()
logistic_params.update({
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
})

logistic_res, _ = run_logistic(data=input_data, params=logistic_params)
logistic_cv = calculate_logloss(input_data["y_train"], logistic_res["oof_preds"])
logger.info(f"Logistic Regression CV Score: {logistic_cv:.4f}")

[2026-08-05 08:14:01] [INFO] --- Logistic Regression 学習開始 ---
[2026-08-05 08:14:01] [INFO] Logistic Regression CV Score: 0.5948


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero enco

## Neural Network テスト

In [7]:
logger.info("--- Neural Network 学習開始 ---")
nn_params = base_params.copy()
nn_params.update({
    "hidden_dims": [256, 128, 64],
    "dropout": 0.3,
    "learning_rate": 0.001,
    "epochs": 50,
    "batch_size": 256,
    "patience": 10,
})

nn_res, _ = run_nn(data=input_data, params=nn_params)
nn_cv = calculate_logloss(input_data["y_train"], nn_res["oof_preds"])
logger.info(f"Neural Network CV Score: {nn_cv:.4f}")

[2026-08-05 08:14:01] [INFO] --- Neural Network 学習開始 ---
[2026-08-05 08:14:03] [INFO] Neural Network CV Score: 0.6153


## 結果まとめ

In [8]:
# 全モデルのスコアをまとめて表示
results_summary = pd.DataFrame({
    "Model": ["TabNet", "HistGradientBoosting", "Logistic Regression", "Neural Network"],
    "CV Score (LogLoss)": [tabnet_cv, histgb_cv, logistic_cv, nn_cv]
})

results_summary = results_summary.sort_values("CV Score (LogLoss)")
logger.info("\n" + "="*50)
logger.info("全モデルの結果一覧:")
logger.info("\n" + str(results_summary))
logger.info("="*50)

print(results_summary)

[2026-08-05 08:14:03] [INFO] 
[2026-08-05 08:14:03] [INFO] 全モデルの結果一覧:
[2026-08-05 08:14:03] [INFO] 
                  Model  CV Score (LogLoss)
2   Logistic Regression            0.594821
3        Neural Network            0.615275
1  HistGradientBoosting            0.631963
0                TabNet            0.665946
[2026-08-05 08:14:03] [INFO] ==================================================
                  Model  CV Score (LogLoss)
2   Logistic Regression            0.594821
3        Neural Network            0.615275
1  HistGradientBoosting            0.631963
0                TabNet            0.665946


## Optuna ハイパーパラメータ最適化テスト

In [9]:
# Optuna関数のインポート
from common.tabnet.tabnet_model_optuna import run_tabnet_optuna
from common.histgb.histgb_model_optuna import run_histgb_optuna
from common.logistic.logistic_model_optuna import run_logistic_optuna
from common.nn.nn_model_optuna import run_nn_optuna

logger.info("=== Optuna ハイパーパラメータ最適化テスト開始 ===")

[2026-08-05 08:14:03] [INFO] === Optuna ハイパーパラメータ最適化テスト開始 ===


### TabNet Optuna

In [10]:
logger.info("--- TabNet Optuna 最適化開始 ---")
tabnet_optuna_params = base_params.copy()
tabnet_optuna_params.update({
    "n_trials": 3,  # クイックテスト用
    "max_epochs": 10,
    "patience": 10,
    "batch_size": 256,
})

tabnet_optuna_res, tabnet_best_params = run_tabnet_optuna(data=input_data, params=tabnet_optuna_params)
logger.info(f"TabNet Optuna Best Score: {tabnet_optuna_res['best_score']:.4f}")
logger.info(f"TabNet Best Params: {tabnet_best_params}")

[2026-08-05 08:14:03] [INFO] --- TabNet Optuna 最適化開始 ---
Stop training because you reached max_epochs = 10 with best_epoch = 8 and best_val_0_logloss = 2.97101


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 10 with best_epoch = 9 and best_val_0_logloss = 2.65304


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 10 with best_epoch = 9 and best_val_0_logloss = 3.38349


...

/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 10 with best_epoch = 9 and best_val_0_logloss = 2.3674


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 10 with best_epoch = 9 and best_val_0_logloss = 2.58566
[2026-08-05 08:14:29] [INFO] TabNet Optuna Best Score: 1.4396
[2026-08-05 08:14:29] [INFO] TabNet Best Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models', 'n_trials': 3, 'max_epochs': 10, 'patience': 10, 'batch_size': 256, 'n_d': 42, 'n_a': 48, 'n_steps': 3, 'gamma': 1.9699098521619942, 'lambda_sparse': 0.0003142880890840109, 'lr': 0.00026587543983272726, 'mask_type': 'entmax'}


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


### HistGradientBoosting Optuna

In [11]:
logger.info("--- HistGradientBoosting Optuna 最適化開始 ---")
histgb_optuna_params = base_params.copy()
histgb_optuna_params.update({
    "n_trials": 3,
})

histgb_optuna_res, histgb_best_params = run_histgb_optuna(data=input_data, params=histgb_optuna_params)
logger.info(f"HistGradientBoosting Optuna Best Score: {histgb_optuna_res['best_score']:.4f}")
logger.info(f"HistGradientBoosting Best Params: {histgb_best_params}")

[2026-08-05 08:14:29] [INFO] --- HistGradientBoosting Optuna 最適化開始 ---
[2026-08-05 08:14:56] [INFO] HistGradientBoosting Optuna Best Score: 0.6047
[2026-08-05 08:14:56] [INFO] HistGradientBoosting Best Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models', 'n_trials': 3, 'learning_rate': 0.012184186502221764, 'max_iter': 447, 'max_depth': 10, 'min_samples_leaf': 72, 'l2_regularization': 1.5320059381854043e-08, 'max_bins': 249}


### Logistic Regression Optuna

In [12]:
logger.info("--- Logistic Regression Optuna 最適化開始 ---")
logistic_optuna_params = base_params.copy()
logistic_optuna_params.update({
    "n_trials": 3,
})

logistic_optuna_res, logistic_best_params = run_logistic_optuna(data=input_data, params=logistic_optuna_params)
logger.info(f"Logistic Regression Optuna Best Score: {logistic_optuna_res['best_score']:.4f}")
logger.info(f"Logistic Regression Best Params: {logistic_best_params}")

[2026-08-05 08:14:56] [INFO] --- Logistic Regression Optuna 最適化開始 ---


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/li

[2026-08-05 08:14:59] [INFO] Logistic Regression Optuna Best Score: 0.5919
[2026-08-05 08:14:59] [INFO] Logistic Regression Best Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models', 'n_trials': 3, 'C': 0.39079671568228824, 'penalty': 'l1'}


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/li

### Neural Network Optuna

In [13]:
logger.info("--- Neural Network Optuna 最適化開始 ---")
nn_optuna_params = base_params.copy()
nn_optuna_params.update({
    "n_trials": 3,
    "epochs": 10,
    "patience": 10,
    "batch_size": 256,
})

nn_optuna_res, nn_best_params = run_nn_optuna(data=input_data, params=nn_optuna_params)
logger.info(f"Neural Network Optuna Best Score: {nn_optuna_res['best_score']:.4f}")
logger.info(f"Neural Network Best Params: {nn_best_params}")

[2026-08-05 08:14:59] [INFO] --- Neural Network Optuna 最適化開始 ---
[2026-08-05 08:15:03] [INFO] Neural Network Optuna Best Score: 0.6085
[2026-08-05 08:15:03] [INFO] Neural Network Best Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/models', 'n_trials': 3, 'epochs': 10, 'patience': 10, 'batch_size': 256, 'learning_rate': 0.003967605077052989, 'dropout': 0.34044600469728353, 'n_layers': 4, 'hidden_dim_0': 64, 'hidden_dim_1': 512, 'hidden_dim_2': 448, 'hidden_dim_3': 128}


### Optuna結果まとめ

In [14]:
# Optuna結果のまとめ
optuna_results_summary = pd.DataFrame({
    "Model": ["TabNet", "HistGradientBoosting", "Logistic Regression", "Neural Network"],
    "Optuna Best Score (LogLoss)": [
        tabnet_optuna_res["best_score"],
        histgb_optuna_res["best_score"],
        logistic_optuna_res["best_score"],
        nn_optuna_res["best_score"]
    ]
})

optuna_results_summary = optuna_results_summary.sort_values("Optuna Best Score (LogLoss)")
logger.info("\n" + "="*50)
logger.info("Optuna最適化結果一覧:")
logger.info("\n" + str(optuna_results_summary))
logger.info("="*50)

print(optuna_results_summary)

[2026-08-05 08:15:03] [INFO] 
[2026-08-05 08:15:03] [INFO] Optuna最適化結果一覧:
[2026-08-05 08:15:03] [INFO] 
                  Model  Optuna Best Score (LogLoss)
2   Logistic Regression                     0.591867
1  HistGradientBoosting                     0.604747
3        Neural Network                     0.608514
0                TabNet                     1.439629
[2026-08-05 08:15:03] [INFO] ==================================================
                  Model  Optuna Best Score (LogLoss)
2   Logistic Regression                     0.591867
1  HistGradientBoosting                     0.604747
3        Neural Network                     0.608514
0                TabNet                     1.439629
